### IMPORTAÇÃO E EXPLORAÇÃO DOS DADOS

In [ ]:
import pandas as pd

df = pd.read_csv("netflix_titles.csv")
df.head()
#por padrão, nos retorna as 5 primeiras linhas do dataset, mas podemos passar um parâmetro para retornar mais ou menos linhas. Ex: df.head(10) retorna as 10 primeiras linhas do dataset.

In [ ]:
df.shape
# nos retorna o número de linhas e colunas do dataset.

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.describe()

# Declaração do problema que queremos solucionar: Dadas as informações de um conteúdo, prever se ele é um Movie ou TV Show.

y = aquilo que queremos prever → type
X = informações que vamos fornecer ao modelo

Para aprender, vamos ignorar duration (min x seasons) e usar release_year (numérica) e rating (categórica) para facilitar.

### features e target

In [ ]:
y = df["type"]                  # alvo
X = df[["release_year", "rating"]]   # features

### Train/Test Split

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier



X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# test_size=0.2 significa que 20% dos dados serão destinados ao conjunto de teste. (80% treinamento)
# random_state=42 fixa a aleatoriedade da divisão. Sem isso, cada execução poderia separar os dados de maneira diferente. Com random_state=42, conseguimos reproduzir a mesma divisão.
# stratify=y mantém a mesma proporção das classes da variável alvo no conjunto de treino e teste, evitando que uma classe fique sub ou superrepresentada na divisão.


### Preprocessing

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", ["release_year"]), #release_year simplesmente continuará sendo utilizada como número, sem nenhuma transformação (passthrough).
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["rating"]) #usamos onehotencoder para lidar com variáveis categóricas 
    ]
)
# obs: Com handle_unknown="ignore", o encoder não dará erro por encontrar no teste uma categoria que não conhecia durante o treino.

### Primeiro modelo: Random Forest

In [ ]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

model.fit(X_train, y_train) #model.fit() = "aprenda com esses dados."

In [ ]:
y_test_pred = model.predict(X_test)

### MÉTRICAS E MATRIZ DE CONFUSÃO

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_test_pred))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_test_pred)

print(cm)

**Acurácia**: De todas as previsões feitas pelo modelo, em quantas ele acertou?
número de previsoes corretas (TRUE POSITIVE + TRUE NEGATIVE) / número total de previsoes (TRUE POSITIVE + TRUE NEGATIVE + FALSE POSITIVE + FALSE NEGATIVE)

**Precisão**: De todas as previsões positivas feitas pelo modelo, quantas realmente eram positivas? 
previsões corretas da classe que me interessa (TRUE POSITIVE) / todas as previsões dessa classe (TRUE POSITIVE + FALSE POSITIVE)

**Recall**: De todas as previsões positivas que o modelo deveria ter feito, quantas ele fez?
quantos positivos o modelo acertou (TRUE POSITIVE) / quantos positivos existem (TRUE POSITIVE + FALSE NEGATIVE)

Análise: O modelo apresenta um desempenho significativamente melhor para classificar Movies do que TV Shows. A principal fonte de erro é a classificação de TV Shows como Movies: 365 TV Shows foram classificados incorretamente como Movies, enquanto apenas 131 Movies foram classificados como TV Shows. Isso sugere um viés do modelo em favor da classe Movie, possivelmente influenciado pelo maior número de Movies no dataset.


### BASELINE
71,85% de acurácia é melhor do que uma estratégia muito simples?

Por exemplo, podemos criar um modelo que simplesmente sempre prevê a classe mais frequente.

O scikit-learn tem exatamente isso:

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

print(accuracy_score(y_test, baseline_pred))

Análise: O modelo melhora pouco em relação ao baseline e apresenta desempenho bastante desigual entre as classes. Embora consiga identificar Movies com bom recall (89%), seu recall para TV Shows é baixo (32%). A matriz de confusão mostra que o principal erro ocorre ao classificar TV Shows como Movies. Isso indica que as features utilizadas até agora (release_year e rating) não são suficientes para distinguir bem as duas classes.

### Entender o efeito das features

Aqui eu faria um experimento que considero muito bom para você aprender.

In [ ]:
#feature engineering / feature selection

### model selection

### CROSS VALIDATION

Precisamos entender o desempenho do modelo separadamente para cada classe. É exatamente isso que o classification_report faz.